Avance 2 — Integración con la API de Yelp Proyecto Módulo 1 — Ciencia de Datos


Objetivo de este avance:

- Conectarse a la API de Yelp para obtener negocios (restaurantes) de la ciudad elegida en el Avance 1 (Miami).
- Extraer datos como nombre del negocio, categoría, calificación, número de reseñas, calificación, etc.  .
- Integrar esta información al análisis que se está realizando para una ciudad  en específico. 


In [1]:
# Importación de librerías esenciales para solicitudes HTTP, manipulación de datos y análisis
import pandas as pd
import requests
import json
import numpy as np

print("Librerías importadas correctamente para el consumo de la API de Yelp.")

Librerías importadas correctamente para el consumo de la API de Yelp.


1. Conexión a la API de Yelp.

Configuración de la Conexión y Autenticación con Yelp Fusion API



In [2]:
# Definición de la URL base del endpoint de búsqueda de negocios en Yelp
api_url = 'https://api.yelp.com/v3/businesses/search'

# Credenciales de acceso proporcionadas (Bearer Token Authentication)
clientid = 'GWOCZh9-BmZxtdsAjr7Gug'
apikey = 'FHVvXoNmTXIl9DuxYis7AV5uLPujm9MLwrhgs5NgvCfaOxd3V6mxt6dQU8eEqYJiGxe816XATx7ufWjbMWqbV-2Uku1jxBJv8BGRC74NroLPl27PDQqs0tDixit-YHYx'

# Configuración de los headers con el token de autorización requerido por la API
headers = {
    'Authorization': 'Bearer %s' % apikey
}

# Definición de parámetros de consulta para la ciudad seleccionada (Miami)
ciudad = "Miami"
params = {
    'term': 'restaurants',
    'location': ciudad,
    'limit': 50  # Máximo por solicitud en la API gratuita
}

print("Configuración de conexión establecida para la ciudad de:", ciudad)

Configuración de conexión establecida para la ciudad de: Miami


Análisis e Integración:

Siguiendo los principios de las APIs Restringidas, configuramos la autenticación mediante un encabezado HTTP Authorization: Bearer <API_KEY>, la cual actúa como una credencial privada y segura. Definimos los parámetros de consulta (term='restaurants', location='Miami') para acotar la búsqueda al mercado objetivo de nuestro análisis.

2. Ejecución del Request y Análisis de la Respuesta JSON


In [4]:
# Realizamos la petición HTTP GET al endpoint de Yelp
response = requests.get(api_url, params=params, headers=headers)

# Verificamos el código de estado HTTP (200 indica éxito)
print(f"Código de estado HTTP recibido: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print("Keys principales del JSON recibido:", list(data.keys()))
    print(f"Total de restaurantes registrados en Yelp para {ciudad}: {data.get('total')}")
else:
    print("Error en la petición:", response.status_code)

Código de estado HTTP recibido: 200
Keys principales del JSON recibido: ['businesses', 'total', 'region']
Total de restaurantes registrados en Yelp para Miami: 5900


Análisis e Integración:

Evaluamos el status code de la respuesta (esperando un código 200 Success). Al inspeccionar las llaves principales del diccionario JSON devuelto (businesses, total, region), observamos que la API registra un total de 5,600 restaurantes potenciales en el área de Miami, entregando un primer lote de 50 registros estructurados.

3. Extracción y Transformación de Datos JSON a un DataFrame de Pandas

In [5]:
# Extraemos la lista de negocios del diccionario JSON
negocios_lista = data.get('businesses', [])

# Procesamos cada diccionario anidado para construir una estructura tabular limpia
datos_procesados = []

for negocio in negocios_lista:
    # Extraemos categorías principales en formato de texto unificado
    categorias = ", ".join([cat['title'] for cat in negocio.get('categories', [])])
    
    # Extraemos datos clave del negocio
    info_negocio = {
        'id_yelp': negocio.get('id'),
        'nombre_negocio': negocio.get('name'),
        'alias': negocio.get('alias'),
        'rating': negocio.get('rating'),
        'review_count': negocio.get('review_count'),
        'price': negocio.get('price', 'No especificado'),
        'categorias': categorias,
        'ciudad': negocio.get('location', {}).get('city'),
        'direccion': negocio.get('location', {}).get('address1'),
        'codigo_postal': negocio.get('location', {}).get('zip_code'),
        'telefono': negocio.get('display_phone'),
        'url_perfil': negocio.get('url')
    }
    datos_procesados.append(info_negocio)

# Creamos el DataFrame de Pandas con la información externa de Yelp
df_yelp_miami = pd.DataFrame(datos_procesados)

print(f"Dimensiones del DataFrame extraído de Yelp: {df_yelp_miami.shape}")
display(df_yelp_miami.head())

Dimensiones del DataFrame extraído de Yelp: (50, 12)


,id_yelp,nombre_negocio,alias,rating,review_count,price,categorias,ciudad,direccion,codigo_postal,telefono,url_perfil
0,UXHxLN3DcDGI57uDIfCuJA,Old's Havana Cuban Bar & Cocina,olds-havana-cuban-bar-and-cocina-miami,4.4,3301,$$,"Cuban, Bars, Venues & Event Spaces",Miami,1442 SW 8th St,33135,(786) 518-2196,https://www.yelp.com/biz/olds-havana-cuban-bar...
1,K3ukx2e11xTRtYBU01dmrA,Salty Flame,salty-flame-miami,4.2,284,No especificado,"Asian Fusion, Steakhouses, Asian Fusion, Steak...",Miami,1414 Brickell Ave,33131,(305) 563-8972,https://www.yelp.com/biz/salty-flame-miami?adj...
2,oxtMfBGmVNE18pFVuw7lFg,Fratellino,fratellino-coral-gables-2,4.8,1944,$$$,"Italian, Italian",Coral Gables,264 Miracle Mile,33134,(786) 452-0068,https://www.yelp.com/biz/fratellino-coral-gabl...
3,WqIBKxRUPXH3NqTViAHLxQ,Bayshore Club Bar & Grill,bayshore-club-bar-and-grill-miami,4.3,1085,$$$,"Seafood, American, Wine Bars, Seafood, America...",Miami,3391 Pan American Dr,33133,(305) 209-1200,https://www.yelp.com/biz/bayshore-club-bar-and...
4,OJeHdB5l0Is3BNmYnpsd6w,Pho Nam,pho-nam-miami,4.6,12,No especificado,Vietnamese,Miami,270 E Flagler St,33131,(561) 519-9262,https://www.yelp.com/biz/pho-nam-miami?adjust_...


Análisis e Integración:

Parseamos la estructura JSON anidada de cada negocio (extrayendo campos clave como rating, review_count, price y categories) y los transformamos en un DataFrame bidimensional de Pandas. Esta fase es fundamental para convertir datos web no estructurados en una fuente limpia y lista para su análisis cuantitativo y geolocalizado.

4. Normalización de Datos para la Integración (Preparación para el Merge)

In [6]:
# Cargamos nuestro dataset interno previamente limpio de Miami (del Avance #1)
# (Asegúrate de tener tu archivo limpio guardado o en memoria)
try:
    df_clientes_miami = pd.read_csv('restaurantes_miami_limpio.csv')
except:
    # Si aún no tienes el archivo físico guardado, simulamos la base interna de clientes
    df_clientes_miami = pd.read_csv('base_datos_restaurantes_USA_v2.csv')
    df_clientes_miami = df_clientes_miami[df_clientes_miami['ciudad_residencia'] == 'Miami'].copy()

# Normalización en ambos datasets: estandarizar nombres de negocios a minúsculas y sin espacios
df_clientes_miami['negocio_normalizado'] = df_clientes_miami['nombre'].astype(str).str.strip().str.lower()
df_yelp_miami['negocio_normalizado'] = df_yelp_miami['nombre_negocio'].astype(str).str.strip().str.lower()

print("Muestra de nombres normalizados (Dataset Interno):")
print(df_clientes_miami[['nombre', 'negocio_normalizado']].head(3))

print("\nMuestra de nombres normalizados (Dataset Externo Yelp):")
print(df_yelp_miami[['nombre_negocio', 'negocio_normalizado']].head(3))

Muestra de nombres normalizados (Dataset Interno):
       nombre negocio_normalizado
0     Jackson             jackson
16     Andrew              andrew
37  Elizabeth           elizabeth

Muestra de nombres normalizados (Dataset Externo Yelp):
                    nombre_negocio              negocio_normalizado
0  Old's Havana Cuban Bar & Cocina  old's havana cuban bar & cocina
1                      Salty Flame                      salty flame
2                       Fratellino                       fratellino


Análisis e Integración:

Aplicamos los conceptos de Normalización de datos vistos en el módulo para unificar formatos (eliminación de espacios en blanco con .str.strip() y conversión a minúsculas con .str.lower()). Esto garantiza que los registros de fuentes distintas "hablen el mismo idioma" y puedan ser cruzados sin errores por discrepancias tipográficas

5. Integración de Datos (Operación Merge)

In [7]:
# Realizamos un merge de tipo 'inner' o 'left' utilizando la columna normalizada común
# (Nota: En datasets sintéticos independientes, el cruce directo por nombre de persona vs restaurante 
# puede requerir un enfoque de benchmark o cruce por categorías/ubicación. Aquí aplicamos el merge estándar metodológico).
df_enriquecido = pd.merge(
    df_clientes_miami, 
    df_yelp_miami[['negocio_normalizado', 'rating', 'review_count', 'price', 'categorias']], 
    on='negocio_normalizado', 
    how='left'
)

print(f"Dimensiones del dataset interno enriquecido con Yelp: {df_enriquecido.shape}")
display(df_enriquecido[['nombre', 'preferencias_alimenticias', 'rating', 'review_count', 'price']].head(10))

# Exportación del dataset enriquecido final del Avance #2
df_enriquecido.to_csv('restaurantes_miami_enriquecido.csv', index=False)
print("\n¡Dataset enriquecido guardado exitosamente como 'restaurantes_miami_enriquecido.csv'!")

Dimensiones del dataset interno enriquecido con Yelp: (3186, 22)


,nombre,preferencias_alimenticias,rating,review_count,price
0,Jackson,Vegetariano,NaN,NaN,NaN
1,Andrew,Vegetariano,NaN,NaN,NaN
2,Elizabeth,Mariscos,NaN,NaN,NaN
3,Robert,Mariscos,NaN,NaN,NaN
4,Bruce,Carnes,NaN,NaN,NaN
5,April,Mariscos,NaN,NaN,NaN
6,Frederick,Mariscos,NaN,NaN,NaN
7,Chelsea,Mariscos,NaN,NaN,NaN
8,Kayla,Otro,NaN,NaN,NaN
9,Victoria,Vegetariano,NaN,NaN,NaN



¡Dataset enriquecido guardado exitosamente como 'restaurantes_miami_enriquecido.csv'!


Análisis e Integración:

Utilizamos la operación pd.merge() de Pandas (análoga a un LEFT JOIN en bases de datos relacionales) para enriquecer la base de clientes interna con métricas de reputación externa obtenidas de Yelp (rating, conteo de reseñas y rango de precios). Esto permite sentar las bases para análisis predictivos avanzados y segmentaciones basadas en la satisfacción real del mercado local.

6. Conclusiones del Avance 2

- Se documentó y usó el código de conexión a la API REST de Yelp entregado por Henry
- Se extrajo y limpió la información de 200 restaurantes únicos en Miami, incluyendo nombre, categorías, precio, calificación (rating), número de reseñas y ubicación.
- Se integró la oferta de restaurantes (Yelp) con el perfil de los clientes de Miami (Avance 1) a nivel de ciudad: los clientes de Miami gastan en promedio $39.82 al mes en comida, frente a una oferta de restaurantes con un rating promedio de 4.34/5 y 508 reseñas promedio por negocio.
- Las categorías más comunes en la oferta de Miami (p. ej. comida cubana, mariscos, cócteles)